# Faster autoscaling on Amazon SageMaker realtime endpoints with inference components (Application Autoscaling)

---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook.

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)


---

In this notebook we show how the new faster autoscaling feature helps scale sagemaker inference endpoints by almost 6x faster than earlier.

We deploy Meta's `Llama3-8B-Instruct` model to an Amazon SageMaker realtime endpoint using Text Generation Inference (TGI) Deep Learning Container (DLC) and apply <span style='color:green'><b>Application Autoscaling</b></span> scaling policies to the endpoint.


<div class="alert alert-block alert-warning">
    Please select <b>m5.2xlarge</b> or larger instance types when running this on Amazon SageMaker Notebook Instance.<br/>
    Select <b>conda_pytorch_p310</b> kernel when running this notebook on Amazon SageMaker Notebook Instance. <br/><br/>
    Ensure python version for the kernel is <b>3.10.x</b> (3.11 is not supported). <br/>
</div>

---

## Prerequisites



<div style="border: 1px solid #f00; border-radius: 5px; padding: 10px; background-color: #fee;">
Before using this notebook please ensure you have access to an active access token from HuggingFace and have accepted the license agreement from Meta.

- **Step 1:** Create user access token in HuggingFace (HF). Refer [here](https://huggingface.co/docs/hub/security-tokens) on how to create HF tokens.
- **Step 2:** Login to [HuggingFace](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/tree/main) and navigate to *Meta-Llama-3-8B-Instruct** home page.
- **Step 3:** Accept META LLAMA 3 COMMUNITY LICENSE AGREEMENT by following the instructions [here](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/tree/main)
- **Step 4:** Wait for the approval email from META (Approval may take any where b/w 1-3 hrs)
</div>

Install packages using uv, an extremely fast python package installer\
Read more about uv here <https://astral.sh/blog/uv>

In [1]:
# [exec-copy] neutralized for papermill (env managed by .v3test-venv)
pass

In [2]:
# [exec-copy] neutralized for papermill (env managed by .v3test-venv)
pass

In [3]:
# [exec-copy] neutralized for papermill (env managed by .v3test-venv)
pass

In [4]:
# [exec-copy] neutralized for papermill (env managed by .v3test-venv)
pass

In [5]:
import sys
import os

import time
import json
from getpass import getpass
import boto3
from rich import print

# SageMaker Python SDK V3 (sagemaker-core)
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris, common_utils
from sagemaker.core.resources import Model, EndpointConfig, Endpoint, InferenceComponent
from sagemaker.core.shapes import (
    ProductionVariant,
    ProductionVariantManagedInstanceScaling,
    ProductionVariantRoutingConfig,
    ContainerDefinition,
    InferenceComponentSpecification,
    InferenceComponentComputeResourceRequirements,
    InferenceComponentRuntimeConfig,
)

[07/16/26 09:28:25] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1697589;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1697590;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


## Initiate sagemaker session

In [6]:
import boto3
from sagemaker.core.helper.session_helper import Session
from sagemaker.core import image_uris, common_utils

region = "us-west-2"
boto_session = boto3.Session(region_name=region)
sess = Session(boto_session=boto_session)
role = "arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045"

sagemaker_client = sess.sagemaker_client
cloudwatch_client = boto3.client("cloudwatch", region_name=region)

hf_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tgi_dlc = image_uris.retrieve(
    framework=image_uris.HUGGING_FACE_LLM_FRAMEWORK,
    region=region,
    version="2.0.0",
)
print('TGI DLC:', tgi_dlc)
print('Region:', region)
print('Role:', role)

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1697595;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1697596;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1697601;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1697602;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Defaulting to only available Python version: py310                   ]8;id=1697609;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=1697610;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#615\615]8;;\

                    INFO     Defaulting to only supported image scope: gpu.                       ]8;id=1697616;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=1697617;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#539\539]8;;\

TGI DLC: 
763104351884.dkr.ecr.us-west-2.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubunt
u22.04

Region: us-west-2

Role: arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045

## Create Endpoint

1. Create `EndpointConfiguration`
2. Create Endpoint

In [7]:
# Set an unique endpoint config name
prefix = common_utils.unique_name_from_base("llama3")
print(f"prefix: {prefix}")

endpoint_config_name = f"{prefix}-endpoint-config"
print(f"Endpoint config name: {endpoint_config_name}")

# Set variant name and instance type for hosting
variant_name = "AllTraffic"
instance_type = "ml.g5.2xlarge"
model_data_download_timeout_in_seconds = 3600
container_startup_health_check_timeout_in_seconds = 3600

initial_instance_count = 1
max_instance_count = 2
print(f"Initial instance count: {initial_instance_count}")
print(f"Max instance count: {max_instance_count}")

# V3: create EndpointConfig via sagemaker-core resource class
endpoint_config = EndpointConfig.create(
    endpoint_config_name=endpoint_config_name,
    execution_role_arn=role,
    production_variants=[
        ProductionVariant(
            variant_name=variant_name,
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=model_data_download_timeout_in_seconds,
            container_startup_health_check_timeout_in_seconds=container_startup_health_check_timeout_in_seconds,
            managed_instance_scaling=ProductionVariantManagedInstanceScaling(
                status="ENABLED",
                min_instance_count=initial_instance_count,
                max_instance_count=max_instance_count,
            ),
            routing_config=ProductionVariantRoutingConfig(
                routing_strategy="LEAST_OUTSTANDING_REQUESTS"
            ),
        )
    ],
)
print(endpoint_config)

prefix: llama3-1784219305-110d

Endpoint config name: llama3-1784219305-110d-endpoint-config

Initial instance count: 1

Max instance count: 2

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


...

                    WARNING  No region provided. Using default region.                                 ]8;id=1697632;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=1697633;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=1697639;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=1697640;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1697645;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1697646;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=1697651;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=1697652;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

EndpointConfig(
    endpoint_config_name='llama3-1784219305-110d-endpoint-config',
    endpoint_config_arn='arn:aws:sagemaker:us-west-2:729646638167:endpoint-config/llama3-1784219305-110d-endpoint-c
onfig',
    production_variants=[
        ProductionVariant(
            variant_name='AllTraffic',
            model_name=Unassigned(),
            initial_instance_count=1,
            instance_type='ml.g5.2xlarge',
            instance_pools=Unassigned(),
            variant_instance_provision_timeout_in_seconds=Unassigned(),
            initial_variant_weight=Unassigned(),
            accelerator_type=Unassigned(),
            core_dump_config=Unassigned(),
            serverless_config=Unassigned(),
            volume_size_in_gb=Unassigned(),
            model_data_download_timeout_in_seconds=3600,
            container_startup_health_check_timeout_in_seconds=3600,
            enable_ssm_access=Unassigned(),
            managed_instance_scaling=ProductionVariantManagedInstanceScaling(
                status='ENABLED',
                min_instance_count=1,
                max_instance_count=2,
                scale_in_policy=Unassigned()
            ),
            routing_config=ProductionVariantRoutingConfig(routing_strategy='LEAST_OUTSTANDING_REQUESTS'),
            inference_ami_version=Unassigned(),
            capacity_reservation_config=Unassigned()
        )
    ],
    data_capture_config=Unassigned(),
    kms_key_id=Unassigned(),
    creation_time=datetime.datetime(2026, 7, 16, 9, 28, 26, 360000, tzinfo=tzlocal()),
    async_inference_config=Unassigned(),
    explainer_config=Unassigned(),
    shadow_production_variants=Unassigned(),
    execution_role_arn='arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045',
    vpc_config=Unassigned(),
    enable_network_isolation=False,
    metrics_config=MetricsConfig(
        enable_enhanced_metrics=Unassigned(),
        metric_publish_frequency_in_seconds=Unassigned()
    )
)

In [8]:
# Set a unique endpoint name
endpoint_name = f"{prefix}-endpoint"

# V3: create Endpoint via sagemaker-core resource class
endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_config_name,
)
print(f"Creating endpoint: [b blue]{endpoint_name}...")
endpoint.wait_for_status("InService")

[07/16/26 09:28:26] INFO     Creating endpoint resource.                                         ]8;id=1697658;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1697659;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10228\10228]8;;\

Creating endpoint: llama3-1784219305-110d-endpoint...

Output()

[07/16/26 09:30:54] INFO     Final Resource Status: InService                                    ]8;id=1697665;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1697666;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10484\10484]8;;\

## Deploy model

Create and deploy model using Amazon SageMaker HuggingFace TGI DLC

<https://sagemaker.readthedocs.io/en/stable/api/inference/model.html#sagemaker.model.Model.deploy>

<div class="alert alert-block alert-warning">
<b>NOTE:</b> Remember to copy your Hugging Face Access Token from <a href="https://hf.co/">https://hf.co/</a> before running the below cell.<br/><br/>
Refer <a href="https://huggingface.co/docs/hub/security-tokens">here</a> to learn about creating HF tokens.
</div>

## Configure container and environment 

In [9]:
# print ecr image uri
print(f"llm image uri: [b green]{tgi_dlc}")

HF_TOKEN = None  # [exec-copy] non-gated model needs no token

llama3model = ContainerDefinition(
    image=tgi_dlc,
    environment={
        "HF_MODEL_ID": "NousResearch/Meta-Llama-3-8B-Instruct",  # [exec-copy] non-gated mirror
        "SM_NUM_GPUS": "1",  # Number of GPU used per replica
        "MAX_INPUT_LENGTH": "2048",  # Max length of input text
        "MAX_TOTAL_TOKENS": "4096",  # Max length of the generation (including input text)
        "MAX_BATCH_TOTAL_TOKENS": "8192",  # Limits the number of tokens processed in parallel
        "MESSAGES_API_ENABLED": "true",  # Enable the messages API
    },
)

# create Model (V3: sagemaker-core resource class)
deployment_name = "sm"
model_name = f"{deployment_name}-model-llama3"

print(f"Creating model: [b green]{model_name}...")
model = Model.create(
    model_name=model_name,
    execution_role_arn=role,
    containers=[llama3model],
)

print(model)

llm image uri: 
763104351884.dkr.ecr.us-west-2.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gpu-py310-cu121-ubunt
u22.04

Creating model: sm-model-llama3...

                    INFO     Creating model resource.                                            ]8;id=1701148;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1701149;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20593\20593]8;;\

Model(
    model_name='sm-model-llama3',
    primary_container=Unassigned(),
    containers=[
        ContainerDefinition(
            container_hostname=Unassigned(),
            image='763104351884.dkr.ecr.us-west-2.amazonaws.com/huggingface-pytorch-tgi-inference:2.1.1-tgi2.0.0-gp
u-py310-cu121-ubuntu22.04',
            image_config=Unassigned(),
            mode='SingleModel',
            model_data_url=Unassigned(),
            model_data_source=Unassigned(),
            additional_model_data_sources=Unassigned(),
            environment={
                'HF_MODEL_ID': 'NousResearch/Meta-Llama-3-8B-Instruct',
                'MAX_BATCH_TOTAL_TOKENS': '8192',
                'MAX_INPUT_LENGTH': '2048',
                'MAX_TOTAL_TOKENS': '4096',
                'MESSAGES_API_ENABLED': 'true',
                'SM_NUM_GPUS': '1'
            },
            model_package_name=Unassigned(),
            inference_specification_name=Unassigned(),
            multi_model_config=Unassigned()
        )
    ],
    inference_execution_config=Unassigned(),
    execution_role_arn='arn:aws:iam::729646638167:role/service-role/AmazonSageMaker-ExecutionRole-20251201T194045',
    vpc_config=Unassigned(),
    creation_time=datetime.datetime(2026, 7, 16, 9, 30, 56, 296000, tzinfo=tzlocal()),
    model_arn='arn:aws:sagemaker:us-west-2:729646638167:model/sm-model-llama3',
    enable_network_isolation=False,
    deployment_recommendation=Unassigned()
)

In [10]:
# Deploy model to Amazon SageMaker Inference Component (V3: sagemaker-core resource class)
inference_component_name_llama3b = f"{prefix}-IC-llama3b"
variant_name = "AllTraffic"

inference_component = InferenceComponent.create(
    inference_component_name=inference_component_name_llama3b,
    endpoint_name=endpoint_name,
    variant_name=variant_name,
    specification=InferenceComponentSpecification(
        model_name=f"{deployment_name}-model-llama3",
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            number_of_accelerator_devices_required=1,
            number_of_cpu_cores_required=1,
            min_memory_required_in_mb=1024,
        ),
    ),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1),
)

# Wait for IC to come InService
print(f"InferenceComponent [b magenta]{inference_component_name_llama3b}...")
inference_component.wait_for_status("InService")
print(inference_component.inference_component_status)

[07/16/26 09:30:56] INFO     Creating inference_component resource.                              ]8;id=1701155;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1701156;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#16568\16568]8;;\

InferenceComponent llama3-1784219305-110d-IC-llama3b...

Output()

[07/16/26 09:35:57] INFO     Final Resource Status: InService                                    ]8;id=1701162;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1701163;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#16816\16816]8;;\

InService

## Inference

Invoke and test endpoint using messages API. Refer to HF [Messages API](https://huggingface.co/docs/text-generation-inference/messages_api) for more info.

In [11]:
# V3 has no Predictor; invoke the inference-component-backed endpoint via
# sagemaker.core.resources.Endpoint.invoke (supports inference_component_name).
def invoke_ic(payload):
    response = endpoint.invoke(
        body=json.dumps(payload).encode("utf-8"),
        content_type="application/json",
        accept="application/json",
        inference_component_name=inference_component_name_llama3b,
    )
    body = response.body
    raw = body.read() if hasattr(body, "read") else body
    return json.loads(raw)

In [12]:
# Prompt to generate
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is deep learning?"},
]

# Generation arguments
parameters = {
    "model": hf_model_id,  # model id is required
    "top_p": 0.6,
    "temperature": 0.9,
    "max_tokens": 512,
    "stop": ["<|eot_id|>"],
}

chat = invoke_ic({"messages": messages, **parameters})

# Unpack and print response
print(chat["choices"][0]["message"]["content"].strip())

Deep learning is a subfield of machine learning that is inspired by the structure and function of the human brain. 
It involves the use of artificial neural networks with multiple layers to analyze and interpret data. These 
networks are designed to mimic the way the brain processes information, with each layer building on the previous 
one to extract increasingly abstract features and representations from the data.

Deep learning models are particularly well-suited to tasks that involve:

1. Image and video recognition: Deep learning models can be trained to recognize objects, scenes, and activities in
images and videos.
2. Natural language processing: Deep learning models can be used to analyze and generate text, speech, and other 
forms of human communication.
3. Speech recognition: Deep learning models can be trained to recognize spoken words and phrases, and to transcribe
them into text.
4. Time series forecasting: Deep learning models can be used to predict future values in a time series, such as 
stock prices or weather patterns.

Some of the key benefits of deep learning include:

1. Ability to learn complex patterns: Deep learning models can learn to recognize complex patterns and 
relationships in data that may not be apparent to humans.
2. Ability to handle large datasets: Deep learning models can be trained on large datasets, and can learn to 
recognize patterns and relationships that may not be apparent in smaller datasets.
3. Ability to generalize to new data: Deep learning models can be trained on a dataset, and then used to make 
predictions on new, unseen data.

However, deep learning also has some challenges and limitations, including:

1. Difficulty in interpreting results: Deep learning models can be difficult to interpret, and it can be 
challenging to understand why a particular model is making a particular prediction.
2. Need for large amounts of data: Deep learning models require large amounts of data to train, which can be 
time-consuming and expensive to collect.
3. Risk of overfitting: Deep learning models can overfit to the training data, which means that they may not 
generalize well to new, unseen data.

Some of the most popular deep learning frameworks and libraries include:

1. TensorFlow
2. PyTorch
3. Keras
4. Caffe
5. Microsoft Cognitive Toolkit (CNTK)

Some of the most popular applications of deep learning include:

1. Self-driving cars
2. Virtual assistants (e.g. Siri, Alexa)
3. Image and video recognition systems
4. Natural language processing systems (e.g. chatbots, language translation)
5. Medical diagnosis and treatment planning systems

I hope this

## Apply Autoscaling policies to the endpoint

Apply Application Autoscaling Policy to endpoint

1. Register Scalable Target

In [13]:
as_min_capacity = 1
as_max_capacity = 2

resource_id = f"inference-component/{inference_component_name_llama3b}"

autoscaling_client = boto3.client("application-autoscaling", region_name=region)

# Register scalable target
scalable_target = autoscaling_client.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:inference-component:DesiredCopyCount",
    MinCapacity=as_min_capacity,
    MaxCapacity=as_max_capacity,  # Replace with your desired maximum instances
)

scalable_target_arn = scalable_target["ScalableTargetARN"]
print(f"Resource ID: [b blue]{resource_id}")
print(f"Scalable_target_arn:\n[b green]{scalable_target_arn}")

Resource ID: inference-component/llama3-1784219305-110d-IC-llama3b

Scalable_target_arn:
arn:aws:application-autoscaling:us-west-2:729646638167:scalable-target/056me97b9c9f67a14d4d99a64d29848b5848

## Use the latest high-resolution Metrics to trigger auto-scaling

- New feature introduces a new <span style='color:green'><b>PredefinedMetricType</b></span> for scaling policy configuration i.e. <span style='color:green'><b>SageMakerVariantConcurrentRequestsPerModelHighResolution</b></span> to trigger scaling actions.
- Creating a scaling policy with this metric type will create cloudwatch alarms that track a new metric called <span style='color:green'><b>ConcurrentRequestsPerModel</b></span>.
- These high-resolution metrics are published at sub-minute intervals (10s intervals to CW + any additional jitter + delays)
- We should observe significant improvement in scale out times with this new metric


### Steps to create Application autoscaling policy

- Create scaling policy
  - Set `PolicyType` to `TargetTrackingScaling`
  - Set `TargetValue` to `5.0`. i.e., Scaling triggers when endpoint receives 5 `ConcurrentRequestsPerModel`
  - Set `PredefinedMetricType` to `SageMakerVariantConcurrentRequestsPerModelHighResolution`
  - Set `ScaleInCoolDown` and `ScaleOutCoolDown` values to `300` seconds

In [14]:
# Create Target Tracking Scaling Policy
target_tracking_policy_response = autoscaling_client.put_scaling_policy(
    PolicyName="SageMakerICScalingPolicy",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:inference-component:DesiredCopyCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 5.0,  # Scaling triggers when endpoint receives 5 ConcurrentRequestsPerModel
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "SageMakerInferenceComponentConcurrentRequestsPerCopyHighResolution"
        },
        "ScaleInCooldown": 300,  # Cooldown period after scale-in activity
        "ScaleOutCooldown": 300,  # Cooldown period after scale-out activity
    },
)

# print(target_tracking_policy_response)
print(f"Policy ARN: [i blue]{target_tracking_policy_response['PolicyARN']}")

# print Cloudwatch Alarms
alarms = target_tracking_policy_response["Alarms"]

for alarm in alarms:
    print(f"[b]Alarm Name:[/b] [b magenta]{alarm['AlarmName']}")
    # print(f"[b]Alarm ARN:[/b] [i green]{alarm['AlarmARN']}[/i green]")
    print("===" * 15)

Policy ARN: 
arn:aws:autoscaling:us-west-2:729646638167:scalingPolicy:e97b9c9f-67a1-4d4d-99a6-4d29848b5848:resource/sagemaker/in
ference-component/llama3-1784219305-110d-IC-llama3b:policyName/SageMakerICScalingPolicy

Alarm Name: 
TargetTracking-inference-component/llama3-1784219305-110d-IC-llama3b-AlarmHigh-c16910ac-96c5-4331-b199-503b8216ed00

=============================================

Alarm Name: 
TargetTracking-inference-component/llama3-1784219305-110d-IC-llama3b-AlarmLow-70d16e46-1517-4809-a603-ed00b68a5b41

=============================================

## Cleanup

- Deregister scalable target. This automatically deletes associated cloudwatch alarms.
- Delete model
- Delete endpoint

In [15]:
try:
    # Deregister the scalable target for AAS (boto3, service-agnostic).
    # NOTE: ScalableDimension must match the one used at registration
    # (inference-component:DesiredCopyCount), fixing a bug in the original notebook.
    autoscaling_client.deregister_scalable_target(
        ServiceNamespace="sagemaker",
        ResourceId=resource_id,
        ScalableDimension="sagemaker:inference-component:DesiredCopyCount",
    )
    print(f"Scalable target for [b]{resource_id}[/b] deregistered. \u2705")
except autoscaling_client.exceptions.ObjectNotFoundException:
    print(f"Scalable target for [b]{resource_id}[/b] not found!.")

print("---" * 10)

try:
    print(f"Deleting inference component: [b magenta]{inference_component_name_llama3b} \u2705")
    inference_component.delete()
except Exception as e:
    print(f"{e}")

try:
    print(f"Deleting model: [b magenta]{model_name} \u2705")
    model.delete()
except Exception as e:
    print(f"{e}")

try:
    print(f"Deleting endpoint: [b magenta]{endpoint_name} \u2705")
    endpoint.delete()
except Exception as e:
    print(f"{e}")

try:
    print(f"Deleting endpoint config: [b magenta]{endpoint_config_name} \u2705")
    endpoint_config.delete()
except Exception as e:
    print(f"{e}")

print("---" * 10)
print("Done")

Scalable target for inference-component/llama3-1784219305-110d-IC-llama3b deregistered. ✅

------------------------------

Deleting inference component: llama3-1784219305-110d-IC-llama3b ✅

[07/16/26 09:36:19] INFO     Deleting InferenceComponent - llama3-1784219305-110d-IC-llama3b     ]8;id=1708219;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1708220;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#16768\16768]8;;\

Deleting model: sm-model-llama3 ✅

...

                    INFO     Deleting Endpoint - llama3-1784219305-110d-endpoint                 ]8;id=1708233;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1708234;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10428\10428]8;;\

Deleting endpoint config: llama3-1784219305-110d-endpoint-config ✅

[07/16/26 09:36:20] INFO     Deleting EndpointConfig - llama3-1784219305-110d-endpoint-config    ]8;id=1708240;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=1708241;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11220\11220]8;;\

------------------------------

Done

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.


![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/deploy_and_monitor|sm-app_autoscaling_realtime_endpoints_inference_components|sm-app_autoscaling_realtime_endpoints_inference_components.ipynb)